# LASSO -- universo local exclusivo (EPH Gran La Plata, sin ICG ni macro nacional)


Corre los cuatro objetivos de modelado que conviven en el repo (`delta_v`, `delta_participacion_pct`, `delta_voto_exit_total_pct` construida localmente por D22, `magnitud_desplazamiento_ideologico`) en los tres niveles.

### Paso 1 -- Carga del panel y universo de columnas EPH candidatas

In [1]:
import sys
import pandas as pd
import numpy as np

general_path = "/workspaces/analisis-politica-economia/"
data_path = f"{general_path}data/tfi_data/"
sys.path.insert(0, f"{general_path}/src")

import ml_models
from ml_models.cargar_panel import cargar_panel, columnas_candidatas
import importlib
import ml_models.lasso
from ml_models.lasso import *
importlib.reload(ml_models.lasso)

NIVELES = ["municipal", "provincial", "nacional"]
paneles = {nivel: cargar_panel(nivel, f"{data_path}panel_ventanas.csv") for nivel in NIVELES}

for nivel, df in paneles.items():
    print(f"{nivel}: {df.shape[0]} filas x {df.shape[1]} columnas")

municipal: 12 filas x 151 columnas
provincial: 12 filas x 151 columnas
nacional: 7 filas x 151 columnas


In [ ]:
PREFIJOS_EPH = [
    "tasa_informalidad", "pct_sin_cobertura_salud", "hacinamiento_medio",
    "pct_hogares_ayuda_social_gobierno", "pct_hogares_prestamo_bancario",
    "pct_hogares_vendio_pertenencias",
]

TARGETS = ["delta_v", "delta_participacion_pct", "delta_voto_exit_total_pct", "magnitud_desplazamiento_ideologico"]

for nivel in NIVELES:
    df = paneles[nivel]
    if "delta_voto_exit_total_pct" not in df.columns:
        df["delta_voto_exit_total_pct"] = df["delta_voto_exit_ausentismo_pct"] + df["delta_voto_exit_blanco_nulo_pct"]

cols_eph_por_nivel = {}
for nivel in NIVELES:
    df = paneles[nivel]
    cols = columnas_candidatas(df, excluir_adicional=TARGETS + ["delta_voto_exit_ausentismo_pct", "delta_voto_exit_blanco_nulo_pct"])
    cols_eph_por_nivel[nivel] = [c for c in cols if any(c.startswith(p) for p in PREFIJOS_EPH)]
    print(f"{nivel}: {len(cols_eph_por_nivel[nivel])} columnas EPH candidatas, N={len(df)}")

municipal: 25 columnas EPH candidatas, N=12
provincial: 25 columnas EPH candidatas, N=12
nacional: 25 columnas EPH candidatas, N=7


/tmp/ipykernel_154293/4141592377.py:15: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["delta_voto_exit_total_pct"] = df["delta_voto_exit_ausentismo_pct"] + df["delta_voto_exit_blanco_nulo_pct"]


### Paso 2 -- Sub-selección: colapsar clusters redundantes (umbral 0.90)

Igual criterio que el resto de los notebooks: la reducción depende solo de la colinealidad entre predictores, no del target, así que se corre una vez por nivel y se reutiliza para los cuatro objetivos. `ORDEN_SUFIJO` prioriza nivel/tendencia de corto plazo por sobre volatilidad/cobertura al elegir representante de cada cluster; no hay `PRIORIDAD_TEORICA` entre variables porque acá todas son EPH, ninguna tiene prioridad teórica declarada sobre las demás en el registro.

In [3]:
UMBRAL_REDUNDANCIA = 0.90
ORDEN_SUFIJO = ["_nivel_vc", "_delta_nivel", "_final_vc", "_pendiente_vc", "_volatilidad_vc", "_cobertura_parcial"]

cols_finales_por_nivel = {}
for nivel in NIVELES:
    df = paneles[nivel]
    cols_eph = cols_eph_por_nivel[nivel]
    corr = df[cols_eph].corr(method="pearson")
    clusters = encontrar_redundantes(corr, UMBRAL_REDUNDANCIA)
    representantes = [elegir_representante(c, df=df, orden_sufijo=ORDEN_SUFIJO) for c in clusters]
    cols_finales_por_nivel[nivel] = sorted(representantes)
    print(f"{nivel}: {len(cols_eph)} -> {len(representantes)} tras colapsar clusters (>= |{UMBRAL_REDUNDANCIA}|)")
    for c in clusters:
        if len(c) > 1:
            rep = elegir_representante(c, df=df, orden_sufijo=ORDEN_SUFIJO)
            print(f"    cluster: {sorted(c)} -> representante: {rep}")

municipal: 25 -> 19 tras colapsar clusters (>= |0.9|)
    cluster: ['hacinamiento_medio_cobertura_parcial', 'pct_hogares_ayuda_social_gobierno_cobertura_parcial', 'pct_hogares_prestamo_bancario_cobertura_parcial', 'pct_hogares_vendio_pertenencias_cobertura_parcial', 'pct_sin_cobertura_salud_cobertura_parcial'] -> representante: hacinamiento_medio_cobertura_parcial
    cluster: ['tasa_informalidad_delta_nivel', 'tasa_informalidad_pendiente_vl'] -> representante: tasa_informalidad_pendiente_vl
    cluster: ['tasa_informalidad_final_vc', 'tasa_informalidad_nivel_vc'] -> representante: tasa_informalidad_nivel_vc
provincial: 25 -> 19 tras colapsar clusters (>= |0.9|)
    cluster: ['hacinamiento_medio_cobertura_parcial', 'pct_hogares_ayuda_social_gobierno_cobertura_parcial', 'pct_hogares_prestamo_bancario_cobertura_parcial', 'pct_hogares_vendio_pertenencias_cobertura_parcial', 'pct_sin_cobertura_salud_cobertura_parcial'] -> representante: hacinamiento_medio_cobertura_parcial
    cluster: ['t

### Paso 3 -- LASSO LOO-CV (1-SE) + ajuste final, por objetivo y nivel


In [4]:
resumen_filas = []

for target in TARGETS:
    print(f"\n{'#'*70}\nOBJETIVO: {target}\n{'#'*70}")
    for nivel in NIVELES:
        df = paneles[nivel]
        cols = cols_finales_por_nivel[nivel]
        X, y = construir_Xy_final(nivel, cols, paneles, target=target)

        n, p = X.shape
        if n < 4:
            print(f"  [{nivel}] N={n} insuficiente para LOO-CV, se salta")
            continue

        baseline = baseline_trivial_loocv(y)
        resultado_cv = lasso_loocv_manual(X, y, n_alphas=50)
        alpha_1se = resultado_cv["alpha_1se"]
        idx_1se = np.argmin(np.abs(resultado_cv["alphas"] - alpha_1se))
        mse_1se = resultado_cv["mean_mse"][idx_1se]

        beta_final = ajustar_final(X, y, alpha_1se)
        activos = beta_final[beta_final != 0].sort_values(key=abs, ascending=False)

        print(f"\n  [{nivel}] N={n}, P={p}  |  MSE trivial(LOO)={baseline:.3f}  MSE LASSO(1-SE)={mse_1se:.3f}  alpha_1se={alpha_1se:.4f}")
        if len(activos):
            print(f"    Coeficientes activos (escala estandarizada): {dict(activos.round(3))}")
        else:
            print("    Ningún coeficiente sobrevive alpha_1se (modelo nulo -- predice la media)")

        frac_seleccion = {}
        if len(activos):
            estab = estabilidad_seleccion(nivel, alpha_1se, df, cols, target, X, y)
            frac_seleccion = (estab[activos.index] != 0).mean().round(2).to_dict()
            print(f"    Estabilidad (frac. de corridas leave-one-transition-out con coef != 0): {frac_seleccion}")

        resumen_filas.append({
            "objetivo": target, "nivel": nivel, "N": n, "P": p,
            "mse_trivial": round(baseline, 3), "mse_lasso_1se": round(mse_1se, 3),
            "mejora_pct": round(100 * (1 - mse_1se / baseline), 1) if baseline else None,
            "n_activos": len(activos),
            "variables_activas": "; ".join(activos.index) if len(activos) else "(ninguna)",
        })


######################################################################
OBJETIVO: delta_v
######################################################################
[municipal] excluye 2 fila(s) por NaN: ['municipal_2001_2003', 'municipal_2003_2005']



  [municipal] N=10, P=19  |  MSE trivial(LOO)=251.482  MSE LASSO(1-SE)=257.115  alpha_1se=8.7922
    Ningún coeficiente sobrevive alpha_1se (modelo nulo -- predice la media)
[provincial] excluye 2 fila(s) por NaN: ['provincial_2001_2003', 'provincial_2003_2005']



  [provincial] N=10, P=19  |  MSE trivial(LOO)=245.537  MSE LASSO(1-SE)=325.631  alpha_1se=6.0758
    Coeficientes activos (escala estandarizada): {'hacinamiento_medio_cobertura_parcial': np.float64(0.0)}
    Estabilidad (frac. de corridas leave-one-transition-out con coef != 0): {'hacinamiento_medio_cobertura_parcial': 0.4}
[nacional] excluye 1 fila(s) por NaN: ['nacional_2011_2013']



  [nacional] N=6, P=15  |  MSE trivial(LOO)=142.013  MSE LASSO(1-SE)=107.923  alpha_1se=5.6705
    Coeficientes activos (escala estandarizada): {'pct_hogares_ayuda_social_gobierno_delta_nivel': np.float64(-2.669), 'tasa_informalidad_pendiente_vc': np.float64(0.579)}
    Estabilidad (frac. de corridas leave-one-transition-out con coef != 0): {'pct_hogares_ayuda_social_gobierno_delta_nivel': 1.0, 'tasa_informalidad_pendiente_vc': 0.67}

######################################################################
OBJETIVO: delta_participacion_pct
######################################################################
[municipal] excluye 2 fila(s) por NaN: ['municipal_2001_2003', 'municipal_2003_2005']



  [municipal] N=10, P=19  |  MSE trivial(LOO)=26.018  MSE LASSO(1-SE)=36.483  alpha_1se=1.8746
    Ningún coeficiente sobrevive alpha_1se (modelo nulo -- predice la media)
[provincial] excluye 2 fila(s) por NaN: ['provincial_2001_2003', 'provincial_2003_2005']


...


  [provincial] N=10, P=19  |  MSE trivial(LOO)=21.243  MSE LASSO(1-SE)=24.001  alpha_1se=2.6191
    Ningún coeficiente sobrevive alpha_1se (modelo nulo -- predice la media)
[nacional] excluye 1 fila(s) por NaN: ['nacional_2011_2013']



  [nacional] N=6, P=15  |  MSE trivial(LOO)=60.900  MSE LASSO(1-SE)=88.485  alpha_1se=3.2210
    Ningún coeficiente sobrevive alpha_1se (modelo nulo -- predice la media)

######################################################################
OBJETIVO: magnitud_desplazamiento_ideologico
######################################################################
[municipal] excluye 2 fila(s) por NaN: ['municipal_2001_2003', 'municipal_2003_2005']



  [municipal] N=10, P=19  |  MSE trivial(LOO)=0.156  MSE LASSO(1-SE)=0.162  alpha_1se=0.2270
    Ningún coeficiente sobrevive alpha_1se (modelo nulo -- predice la media)
[provincial] excluye 2 fila(s) por NaN: ['provincial_2001_2003', 'provincial_2003_2005']



  [provincial] N=10, P=19  |  MSE trivial(LOO)=0.173  MSE LASSO(1-SE)=0.128  alpha_1se=0.0565
    Coeficientes activos (escala estandarizada): {'tasa_informalidad_pendiente_vc': np.float64(-0.207), 'tasa_informalidad_nivel_vl': np.float64(-0.178), 'pct_hogares_vendio_pertenencias_nivel_vc': np.float64(-0.032), 'pct_hogares_ayuda_social_gobierno_nivel_vc': np.float64(-0.032)}
    Estabilidad (frac. de corridas leave-one-transition-out con coef != 0): {'tasa_informalidad_pendiente_vc': 1.0, 'tasa_informalidad_nivel_vl': 0.9, 'pct_hogares_vendio_pertenencias_nivel_vc': 0.6, 'pct_hogares_ayuda_social_gobierno_nivel_vc': 0.6}
[nacional] excluye 1 fila(s) por NaN: ['nacional_2011_2013']



  [nacional] N=6, P=15  |  MSE trivial(LOO)=0.032  MSE LASSO(1-SE)=0.035  alpha_1se=0.1310
    Ningún coeficiente sobrevive alpha_1se (modelo nulo -- predice la media)


### Paso 4 -- Resumen

In [5]:
resumen = pd.DataFrame(resumen_filas)
pd.set_option("display.max_colwidth", 200)
pd.set_option("display.width", 200)
resumen

,objetivo,nivel,N,P,mse_trivial,mse_lasso_1se,mejora_pct,n_activos,variables_activas
0,delta_v,municipal,10,19,251.482,257.115,-2.2,0,(ninguna)
1,delta_v,provincial,10,19,245.537,325.631,-32.6,1,hacinamiento_medio_cobertura_parcial
2,delta_v,nacional,6,15,142.013,107.923,24.0,2,pct_hogares_ayuda_social_gobierno_delta_nivel; tasa_informalidad_pendiente_vc
3,delta_participacion_pct,municipal,10,19,26.018,36.483,-40.2,0,(ninguna)
4,delta_participacion_pct,provincial,10,19,27.628,38.677,-40.0,1,tasa_informalidad_volatilidad_vl
5,delta_participacion_pct,nacional,6,15,54.286,79.757,-46.9,0,(ninguna)
6,delta_voto_exit_total_pct,municipal,10,19,16.999,19.019,-11.9,0,(ninguna)
7,delta_voto_exit_total_pct,provincial,10,19,21.243,24.001,-13.0,0,(ninguna)
8,delta_voto_exit_total_pct,nacional,6,15,60.900,88.485,-45.3,0,(ninguna)
9,magnitud_desplazamiento_ideologico,municipal,10,19,0.156,0.162,-4.0,0,(ninguna)


### Notas de lectura

- En la corrida exploratoria (16/09/2026) 10 de las 12 combinaciones nivel×objetivo empeoran respecto del baseline trivial -- con este universo exclusivamente local, el modelo no encuentra señal utilizable ahí.
- Dos excepciones con estabilidad alta en leave-one-transition-out: `delta_v` nacional (N=6, `pct_hogares_ayuda_social_gobierno_delta_nivel` con estabilidad 1.0) y `magnitud_desplazamiento_ideologico` provincial (N=10, `tasa_informalidad_pendiente_vc` con estabilidad 1.0).
- Lectura sustantiva de `pct_hogares_ayuda_social_gobierno_delta_nivel`: el registro de variables la documenta como proxy de estrés económico del hogar, no como señal de generosidad de política pública (misma familia que `pct_hogares_prestamo_bancario`/`pct_hogares_vendio_pertenencias`) -- la lectura consistente con esa definición es que la proporción de hogares que recurre a ayuda social funciona como termómetro de cómo se vive la crisis a nivel doméstico, independiente de cómo el oficialismo de turno perciba su propia política de asistencia; es ese malestar el que se correlaciona con la caída del voto oficialista (H1), no la ayuda en sí como causa.
- Caveat central: incluso después de colapsar por colinealidad, P (15-19) sigue por encima de N (6-10) en los tres niveles -- un régimen más chico que el N=17 de Sinha et al. (2024) ya señalado como frontera de fragilidad para LASSO en este proyecto. Tratar como hipótesis a seguir mirando con próximas elecciones, no como hallazgo cerrado.